# Chapter 2 v2 — NB 01: extract PHECODE phenotypes for the 11K (Freeze One / Park cohort)

**Goal:** build the **phecode** phenotype table for the ~11K WES participants (Freeze One) using the **canonical PheWAS method** (Park's approach) — ICD-9/ICD-10 → phecode (Phecode Map 1.2) + **rule-of-2** + **control exclusions** + sex restriction — via the R `PheWAS` package (`createPhenotypes`).

This replaces our earlier raw-ICD tinnitus definition (chapter_2) with the **proper phecode**, so we can (a) match Park's phenotyping and (b) revisit the 4-vs-8 carrier-case question.

**ID chain:** WES `GENO_ID` (`.fam`) → `Demographics` → `PT_ID` → `Diagnosis` (ICD codes). Phenotype lives on `PT_ID`.

In [1]:
# --- Setup ---
import subprocess
from pathlib import Path
import pandas as pd

BASE = Path("/project/hall/analysis/hearing-loss-genomics")
R2   = BASE / "analysis/chapter_2_v2/results"; R2.mkdir(parents=True, exist_ok=True)
FZ   = Path("/static/PMBB/PMBB_Freeze17")
FAM  = FZ / "genotype/exome/all_variants/UPENN_Freeze_One_GRCh38.GL.pVCF.biallelic.fam"
DEMO = FZ / "phenotype/PMBB_Geno_Demographics_Deidentified_012020.csv"
DIAG = FZ / "phenotype/PMBB_Geno_Nonsensitive_Diagnosis_Deidentified_012020.csv"
RSCRIPT  = "/appl/R-4.4/bin/Rscript"
PHEWAS_R = str(BASE / "analysis/chapter_2_v2/scripts/run_phewas_createphenotypes.R")
print("ready")

ready


## 1. The 11K WES cohort + crosswalk to phenotype ID
Phenotype is per-person (`PT_ID`); a few `PT_ID`s map to >1 `GENO_ID` (re-sequenced), so the phenotypable
population is defined at the **distinct `PT_ID`** level.

In [2]:
fam  = pd.read_csv(FAM, sep=r"\s+", header=None, usecols=[1], names=["GENO_ID"])
demo = pd.read_csv(DEMO, dtype=str).drop_duplicates("GENO_ID")[["GENO_ID","PT_ID","GENDER_CODE"]]
cohort = fam.merge(demo, on="GENO_ID", how="left")
print("WES GENO_IDs (Freeze One):", len(cohort))
print("  linked to a PT_ID:", int(cohort.PT_ID.notna().sum()),
      "| unlinked (no phenotype):", int(cohort.PT_ID.isna().sum()))
wes = cohort.dropna(subset=["PT_ID"]).copy()
print("  distinct PT_IDs (phenotypable population):", wes.PT_ID.nunique())

WES GENO_IDs (Freeze One): 11451
  linked to a PT_ID: 9322 | unlinked (no phenotype): 2129
  distinct PT_IDs (phenotypable population): 9249


## 2. Build the PheWAS input from ICD diagnoses — **one row per (id, code, date)**
⚠️ **Format matters.** `createPhenotypes` expects the 4th column (`index`) to be the **raw diagnosis date**,
kept as a **string**. Its rule-of-2 then counts **distinct dates itself**
(`default_code_agg` → `length(unique(index))` for character index).

Do **not** pre-aggregate to an integer count: for a *numeric* index `default_code_agg` switches to `sum()`, and
`mapCodesToPhecodes`' `distinct()` collapses identical `(id, phecode, count)` rows across the several ICD codes
that map to one phecode — which **undercounts** cases (e.g. tinnitus 389.4 came out 114 pre-aggregated vs the
correct **131** with raw dates; hearing-loss 389 came out 357 vs **481**).

In [3]:
diag = pd.read_csv(DIAG, header=None, usecols=[0,1,2,6],
                   names=["PT_ID","CODE","VER","DATE"], dtype=str)
diag = diag[diag.PT_ID.isin(set(wes.PT_ID))]
diag["vocabulary_id"] = diag["VER"].map({"ICD-9":"ICD9CM","ICD-10":"ICD10CM"})
diag = diag.dropna(subset=["vocabulary_id","CODE","DATE"])

# correct form: id, vocabulary_id, code, index(=raw DATE string); one row per distinct (id,code,date)
inp = (diag[["PT_ID","vocabulary_id","CODE","DATE"]]
         .rename(columns={"PT_ID":"id","CODE":"code","DATE":"index"})
         .drop_duplicates())
inp[["id","vocabulary_id","code","index"]].to_csv(R2/"phewas_input.csv", index=False)

wes.rename(columns={"PT_ID":"id","GENDER_CODE":"sex"})[["id","sex"]].dropna().drop_duplicates("id").to_csv(R2/"phewas_sex.csv", index=False)
wes["PT_ID"].drop_duplicates().to_csv(R2/"phewas_pop_ids.txt", index=False, header=False)
print("PheWAS input rows (id×code×date):", len(inp), "| participants:", inp.id.nunique())

PheWAS input rows (id×code×date): 1936110 | participants: 9179


## 2b. Diagnostic — rule-of-2 impact on the ear / hearing-loss phecodes
For every ear/HL phecode: how many participants have the phecode-mapped codes on **0**, **exactly 1**, or **≥2**
distinct dates. The **≥2** column *is* the case count (rule-of-2), and the **1** column is exactly who the rule
drops. Totals always sum to the full cohort. Built by replicating `createPhenotypes`' steps in pandas
(ICD→phecode via Phecode Map 1.2 → **rollup** child→parent → distinct dates per person×phecode), so **≥2**
matches the `createPhenotypes` case counts (§4) — exact for leaf phecodes, ±a few for parents (cohort-N edges).

In [4]:
# export the PheWAS maps once (used here and reusable)
rexport = f'''suppressMessages(library(PheWAS))
write.csv(PheWAS::phecode_map,          "{R2}/phecode_map12.csv",       row.names=FALSE)
write.csv(PheWAS::phecode_rollup_map,   "{R2}/phecode_rollup_map.csv",  row.names=FALSE)
write.csv(PheWAS::pheinfo[,c("phecode","description")], "{R2}/pheinfo.csv", row.names=FALSE)'''
subprocess.run([RSCRIPT, "-e", rexport], check=True, capture_output=True, text=True)

pmap = pd.read_csv(R2/"phecode_map12.csv", dtype=str)          # vocabulary_id, code, phecode
roll = pd.read_csv(R2/"phecode_rollup_map.csv", dtype=str)     # code, phecode_unrolled
pinfo= pd.read_csv(R2/"pheinfo.csv", dtype=str); names=dict(zip(pinfo.phecode,pinfo.description))

dm = (diag.merge(pmap, left_on=["vocabulary_id","CODE"], right_on=["vocabulary_id","code"], how="inner")
          .merge(roll, left_on="phecode", right_on="code", how="inner"))       # child -> parent rollup
occ = dm.groupby(["PT_ID","phecode_unrolled"])["DATE"].nunique().reset_index(name="n_dates")

N = wes.PT_ID.nunique()
EAR = ["388","389","389.1","389.2","389.3","389.4","389.5"]
rows=[]
for pc in EAR:
    s = occ[occ.phecode_unrolled==pc]
    n1 = int((s.n_dates==1).sum()); n2 = int((s.n_dates>=2).sum())
    rows.append({"phecode":pc,"description":names.get(pc,""),
                 "0_visits":N-n1-n2,"1_visit":n1,"2plus_CASE":n2,"total":N})
tab = pd.DataFrame(rows)
tab.to_csv(R2/"ruleof2_ear_phecodes.csv", index=False)
print(f"cohort N = {N}\n")
print(tab.to_string(index=False))

cohort N = 9249

phecode                                description  0_visits  1_visit  2plus_CASE  total
    388                     Other disorders of ear      9185       58           6   9249
    389                               Hearing loss      8375      397         477   9249
  389.1                 Sensorineural hearing loss      8750      274         225   9249
  389.2                    Conductive hearing loss      9158       61          30   9249
  389.3 Degenerative and vascular disorders of ear      9232       14           3   9249
  389.4                                   Tinnitus      8958      160         131   9249
  389.5                Disorders of acoustic nerve      9216       30           3   9249


## 2c. Sensitivity — count **events** instead of **distinct dates**
What if Park counted the *number of code events* (raw diagnosis rows) rather than *distinct dates*? Then a code
repeated twice **on the same day** would satisfy the rule-of-2. This is a looser bar (≥ the §2b counts). We tabulate
it to bound how much the date-vs-event choice can move the phenotype — and later (NB02) whether it moves the ZNF175
carrier-cases off 4.

In [5]:
# events = raw diagnosis rows mapping to the (rolled-up) phecode, NOT deduped to distinct dates
occ_ev = dm.groupby(["PT_ID","phecode_unrolled"]).size().reset_index(name="n_events")
rows=[]
for pc in EAR:
    s = occ_ev[occ_ev.phecode_unrolled==pc]
    n1=int((s.n_events==1).sum()); n2=int((s.n_events>=2).sum())
    rows.append({"phecode":pc,"description":names.get(pc,""),
                 "0_events":N-n1-n2,"1_event":n1,"2plus_events":n2,"total":N})
tab_ev = pd.DataFrame(rows); tab_ev.to_csv(R2/"ruleof2_ear_phecodes_events.csv", index=False)
print(tab_ev.to_string(index=False))
mg = tab.merge(tab_ev, on="phecode")[["phecode","description_x","2plus_CASE","2plus_events"]]
mg["extra_if_events"] = mg["2plus_events"] - mg["2plus_CASE"]
print("\nExtra cohort cases if counting events vs distinct dates (≥2):")
print(mg.rename(columns={"description_x":"description"}).to_string(index=False))

phecode                                description  0_events  1_event  2plus_events  total
    388                     Other disorders of ear      9185       58             6   9249
    389                               Hearing loss      8375      276           598   9249
  389.1                 Sensorineural hearing loss      8750      176           323   9249
  389.2                    Conductive hearing loss      9158       58            33   9249
  389.3 Degenerative and vascular disorders of ear      9232       14             3   9249
  389.4                                   Tinnitus      8958      139           152   9249
  389.5                Disorders of acoustic nerve      9216       27             6   9249

Extra cohort cases if counting events vs distinct dates (≥2):
phecode                                description  2plus_CASE  2plus_events  extra_if_events
    388                     Other disorders of ear           6             6                0
    389              

## 2d. Genotype bridge — the §2b view + ZNF175 carrier columns
Each ear/HL phecode row now carries the genotype side too. Among that phecode's **cases** (rule-of-2), how many carry:
- **any_ZNF175** — ≥1 ZNF175 variant of *any* frequency/consequence (mostly common polymorphisms → uninformative baseline),
- **rare_ZNF175** — ≥1 *rare* ZNF175 variant (cohort MAF ≤ 0.1%, any consequence),
- **qual_pLOF** — ≥1 *qualifying* rare **pLOF** variant (our analysis set, `chapter_2/results/06/carriers_v1.csv`).

The `qual_pLOF` column is the ZNF175→tinnitus signal: **389.4 → 4**. Note how the pLOF restriction narrows rare→qualifying
(e.g. tinnitus 11 rare-carriers → 4 pLOF-carriers). Genotype from `chapter_2/results/02/v1_znf175_strict_region.vcf.gz`
(161 variants × 11,451 samples); MAF computed from the genotypes (AC/AN).

In [6]:
import subprocess
from collections import defaultdict
BCFTOOLS = "/appl/bcftools-1.21/bin/bcftools"
VZ   = BASE/"analysis/chapter_2/results/02/v1_znf175_strict_region.vcf.gz"
CARR = BASE/"analysis/chapter_2/results/06/carriers_v1.csv"

q = subprocess.run([BCFTOOLS,"query","-f","[%SAMPLE\t%CHROM:%POS:%REF:%ALT\t%GT\n]",str(VZ)],
                   capture_output=True, text=True).stdout
ac=defaultdict(int); an=defaultdict(int); salt=defaultdict(list)
for ln in q.split("\n"):
    if not ln: continue
    s,vid,gt = ln.split("\t")
    a = gt.count("1"); an[vid]+=gt.count("0")+a; ac[vid]+=a
    if a>0: salt[s].append(vid)
maf = {v:(min(ac[v]/an[v],1-ac[v]/an[v]) if an[v] else 0) for v in an}
rare_ids = {v for v,m in maf.items() if 0<m<=0.001}
any_c  = {s for s,vs in salt.items() if vs}
rare_c = {s for s,vs in salt.items() if any(v in rare_ids for v in vs)}
qual_c = set(pd.read_csv(CARR).query("carrier==1").IID)

g2p = dict(zip(wes.GENO_ID, wes.PT_ID)); cohort_pt = set(wes.PT_ID)
to_pt = lambda ss: {g2p[s] for s in ss if s in g2p} & cohort_pt
anyP, rareP, qualP = to_pt(any_c), to_pt(rare_c), to_pt(qual_c)
print(f"ZNF175 variants: {len(maf)} | rare(MAF<=0.1%): {len(rare_ids)} | "
      f"cohort carriers -> any:{len(anyP)} rare:{len(rareP)} qual-pLOF:{len(qualP)}\n")

rows=[]
for pc in EAR:
    g = dm[dm.phecode_unrolled==pc].groupby("PT_ID")["DATE"].nunique(); cases=set(g[g>=2].index)
    rows.append({"phecode":pc,"description":names.get(pc,""),"cases":len(cases),
                 "any_ZNF175":len(cases&anyP),"rare_ZNF175":len(cases&rareP),"qual_pLOF":len(cases&qualP)})
bridge = pd.DataFrame(rows); bridge.to_csv(R2/"phecode_x_znf175_genotype.csv", index=False)
print(bridge.to_string(index=False))

ZNF175 variants: 161 | rare(MAF<=0.1%): 142 | cohort carriers -> any:7320 rare:287 qual-pLOF:27



phecode                                description  cases  any_ZNF175  rare_ZNF175  qual_pLOF
    388                     Other disorders of ear      6           6            0          0
    389                               Hearing loss    477         384           20          4
  389.1                 Sensorineural hearing loss    225         179           12          2
  389.2                    Conductive hearing loss     30          24            0          0
  389.3 Degenerative and vascular disorders of ear      3           2            0          0
  389.4                                   Tinnitus    131         112           11          4
  389.5                Disorders of acoustic nerve      3           3            0          0


### Reading — phenotypic **specificity** of the ZNF175 signal
The `qual_pLOF` carrier-cases for hearing-loss (389) and tinnitus (389.4) are the **same 4 people**
(PT_IDs `1287154255, 1289754594, 1291558774, 3514664072`). Since 389.4 rolls up into 389, every tinnitus case is
also a 389 case — and because 389 (=4) does **not exceed** 389.4 (=4), **no ZNF175 carrier has hearing loss without
tinnitus**. The 2 sensorineural (389.1) carriers are a *subset* of these 4 (tinnitus + sensorineural). So the ZNF175
carrier signal is **specific to tinnitus**, not diffuse hearing loss — phenotypically coherent with Park (2021).
Were the signal generic ear disease, carriers would scatter across 389.1/389.2 *outside* tinnitus; they do not.

## 3. Run PheWAS `createPhenotypes` (ICD→phecode + rule-of-2 + control exclusions)

In [7]:
out = R2 / "phecodes_11k.csv"
cmd = [RSCRIPT, PHEWAS_R, str(R2/"phewas_input.csv"), str(R2/"phewas_sex.csv"),
       str(R2/"phewas_pop_ids.txt"), str(out)]
print(" ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-600:])
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-2000:]); raise RuntimeError("PheWAS failed")

/appl/R-4.4/bin/Rscript /project/hall/analysis/hearing-loss-genomics/analysis/chapter_2_v2/scripts/run_phewas_createphenotypes.R /project/hall/analysis/hearing-loss-genomics/analysis/chapter_2_v2/results/phewas_input.csv /project/hall/analysis/hearing-loss-genomics/analysis/chapter_2_v2/results/phewas_sex.csv /project/hall/analysis/hearing-loss-genomics/analysis/chapter_2_v2/results/phewas_pop_ids.txt /project/hall/analysis/hearing-loss-genomics/analysis/chapter_2_v2/results/phecodes_11k.csv


wrote /project/hall/analysis/hearing-loss-genomics/analysis/chapter_2_v2/results/phecodes_11k.csv - 9249 participants x 1854 phecodes



## 4. Summarize + key phecodes — should match §2b's **≥2** column
**Tinnitus = phecode 389.4** (NOT 389.2, conductive HL). 389.4 maps from ICD-9 388.3x / ICD-10 H93.1x — the same codes we used in chapter_2.
TRUE = case, FALSE = control, NA = excluded (control-exclusion / insufficient counts / sex).

In [8]:
ph = pd.read_csv(out, dtype=str)
n_phe = ph.shape[1] - 1
print(f"phecode matrix: {ph.shape[0]} participants x {n_phe} phecodes")

def counts(col):
    s = ph[col].astype(str).str.upper()
    return int((s=="TRUE").sum()), int((s=="FALSE").sum()), int(len(ph)-(s=="TRUE").sum()-(s=="FALSE").sum())

for pc,name in [("389.4","TINNITUS"),("389","hearing loss"),("389.1","sensorineural HL"),("389.2","conductive HL")]:
    if pc in ph.columns:
        ca,co,ex = counts(pc)
        print(f"  phecode {pc:6s} ({name:18s}): cases={ca}, controls={co}, excluded/NA={ex}")
    else:
        print(f"  phecode {pc} ({name}) — not present (no cases)")

phecode matrix: 9249 participants x 1854 phecodes
  phecode 389.4  (TINNITUS          ): cases=131, controls=8346, excluded/NA=772
  phecode 389    (hearing loss      ): cases=477, controls=8346, excluded/NA=426
  phecode 389.1  (sensorineural HL  ): cases=225, controls=8346, excluded/NA=678
  phecode 389.2  (conductive HL     ): cases=30, controls=8346, excluded/NA=873


## 5. Attach GENO_ID (link back to genotype/carriers) + save

In [9]:
ph = ph.rename(columns={"id":"PT_ID"})
ph["PT_ID"] = ph["PT_ID"].astype(str)
key = wes[["GENO_ID","PT_ID"]].copy(); key["PT_ID"] = key["PT_ID"].astype(str)
phg = key.merge(ph, on="PT_ID", how="right")
phg.to_csv(R2/"phecodes_11k_with_genoid.csv", index=False)
print("saved -> results/phecodes_11k_with_genoid.csv  shape", phg.shape)
print("keyed by GENO_ID + PT_ID -> joins directly to our ZNF175 carrier lists")

saved -> results/phecodes_11k_with_genoid.csv  shape (9322, 1856)
keyed by GENO_ID + PT_ID -> joins directly to our ZNF175 carrier lists
